# 🐍 Day 7: Mini-Project - Validation Framework

- Decorator-based field validation
- Type validation
- Range and pattern validation
- Complete runnable framework

## Project Overview

We'll build a validation framework that:
- Uses decorators to declare validation rules on class attributes
- Validates on assignment (via descriptors)
- Supports: type check, range (min/max), regex pattern, custom validators

## Step 1: Base Validator Descriptor

Create a descriptor that stores values and can be extended with validation logic.

In [ ]:
class Validated:
    def __init__(self, name=None):
        self.name = name

    def __set_name__(self, owner, name):
        if self.name is None:
            self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        self.validate(value)
        obj.__dict__[self.name] = value

    def validate(self, value):
        pass  # Override in subclasses

## Step 2: Typed Validator

Enforce type on assignment.

In [ ]:
class Typed(Validated):
    def __init__(self, expected_type, name=None):
        super().__init__(name)
        self.expected_type = expected_type

    def validate(self, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name!r} must be {self.expected_type.__name__}")

class Person:
    name = Typed(str)
    age = Typed(int)

p = Person()
p.name = "Alice"
p.age = 30
print(p.name, p.age)
# p.age = "thirty"  # TypeError

## Step 3: Range Validator

Enforce min/max for numeric values.

In [ ]:
class Range(Validated):
    def __init__(self, min_val=None, max_val=None, name=None):
        super().__init__(name)
        self.min_val = min_val
        self.max_val = max_val

    def validate(self, value):
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name!r} must be >= {self.min_val}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name!r} must be <= {self.max_val}")

class Score:
    value = Range(0, 100)

s = Score()
s.value = 85
print(s.value)
# s.value = 150  # ValueError

## Step 4: Pattern Validator

Validate strings against a regex pattern.

In [ ]:
import re

class Pattern(Validated):
    def __init__(self, pattern, name=None):
        super().__init__(name)
        self.pattern = re.compile(pattern)

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name!r} must be str")
        if not self.pattern.match(value):
            raise ValueError(f"{self.name!r} does not match pattern {self.pattern.pattern!r}")

class Email:
    address = Pattern(r'^[\w.-]+@[\w.-]+\.\w+$')

e = Email()
e.address = "user@example.com"
print(e.address)
# e.address = "invalid"  # ValueError

## Step 5: Length Validator

Validate length of strings/lists.

In [ ]:
class Length(Validated):
    def __init__(self, min_len=None, max_len=None, name=None):
        super().__init__(name)
        self.min_len = min_len
        self.max_len = max_len

    def validate(self, value):
        n = len(value)
        if self.min_len is not None and n < self.min_len:
            raise ValueError(f"{self.name!r} length must be >= {self.min_len}")
        if self.max_len is not None and n > self.max_len:
            raise ValueError(f"{self.name!r} length must be <= {self.max_len}")

class Username:
    name = Length(3, 20)

u = Username()
u.name = "alice"
print(u.name)
# u.name = "ab"  # ValueError

## Step 6: Combining Validators with a Decorator

Use a decorator to apply multiple validators to a field. We'll use a simple approach: chain validation in one descriptor.

In [ ]:
def validated(*validators):
    def decorator(func):
        class Composite(Validated):
            def validate(self, value):
                for v in validators:
                    v.validate(value)
        return Composite()
    return decorator

# Simpler: use multiple descriptors
class TypedRange(Validated):
    def __init__(self, expected_type, min_val=None, max_val=None, name=None):
        super().__init__(name)
        self.expected_type = expected_type
        self.min_val = min_val
        self.max_val = max_val

    def validate(self, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name!r} must be {self.expected_type.__name__}")
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name!r} must be >= {self.min_val}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name!r} must be <= {self.max_val}")

class Product:
    price = TypedRange(float, 0, 10000)

pr = Product()
pr.price = 19.99
print(pr.price)

## Step 7: Complete Validation Framework

All validators together with a real-world model.

In [ ]:
import re

class Validated:
    def __init__(self, name=None):
        self.name = name

    def __set_name__(self, owner, name):
        if self.name is None:
            self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        self.validate(value)
        obj.__dict__[self.name] = value

    def validate(self, value):
        pass

class Typed(Validated):
    def __init__(self, expected_type, name=None):
        super().__init__(name)
        self.expected_type = expected_type

    def validate(self, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name!r} must be {self.expected_type.__name__}")

class Range(Validated):
    def __init__(self, min_val=None, max_val=None, name=None):
        super().__init__(name)
        self.min_val, self.max_val = min_val, max_val

    def validate(self, value):
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name!r} must be >= {self.min_val}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name!r} must be <= {self.max_val}")

class Pattern(Validated):
    def __init__(self, pattern, name=None):
        super().__init__(name)
        self.pattern = re.compile(pattern)

    def validate(self, value):
        if not isinstance(value, str) or not self.pattern.match(value):
            raise ValueError(f"{self.name!r} invalid format")

class Length(Validated):
    def __init__(self, min_len=None, max_len=None, name=None):
        super().__init__(name)
        self.min_len, self.max_len = min_len, max_len

    def validate(self, value):
        n = len(value)
        if self.min_len is not None and n < self.min_len:
            raise ValueError(f"{self.name!r} length must be >= {self.min_len}")
        if self.max_len is not None and n > self.max_len:
            raise ValueError(f"{self.name!r} length must be <= {self.max_len}")

In [ ]:
class User:
    username = Length(3, 20)
    email = Pattern(r'^[\w.-]+@[\w.-]+\.\w+$')
    age = Typed(int)
    age_valid = Range(0, 150)  # Need combined - simplified: use Typed + Range in one

# Simpler: TypedRange
class TypedRange(Validated):
    def __init__(self, expected_type, min_val=None, max_val=None, name=None):
        super().__init__(name)
        self.expected_type, self.min_val, self.max_val = expected_type, min_val, max_val

    def validate(self, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name!r} must be {self.expected_type.__name__}")
        Range(self.min_val, self.max_val, self.name).validate(value)

class User:
    username = Length(3, 20)
    email = Pattern(r'^[\w.-]+@[\w.-]+\.\w+$')
    age = TypedRange(int, 0, 150)

u = User()
u.username = "alice"
u.email = "alice@example.com"
u.age = 25
print(u.username, u.email, u.age)

## Step 8: Run the Demo

Test valid and invalid inputs.

In [ ]:
u = User()
try:
    u.username = "ab"  # Too short
except ValueError as e:
    print(f"Caught: {e}")

try:
    u.email = "not-an-email"
except ValueError as e:
    print(f"Caught: {e}")

try:
    u.age = 200
except ValueError as e:
    print(f"Caught: {e}")

u.username = "alice"
u.email = "alice@test.com"
u.age = 30
print("Valid user:", u.username, u.email, u.age)

## Extensions to Try

- Add OneOf (value in allowed set)
- Add custom validator function decorator
- Add validate_all() to validate entire instance
- Add to_dict() / from_dict() for serialization

## Key Takeaways

- Descriptors + __set_name__ for declarative validation
- Validators run on assignment
- Combine validators (Typed + Range) for complex rules
- Pattern and Length for string validation

## 🎉 Week 9 Complete!

You've mastered decorators, descriptors, metaclasses, and dynamic attributes. Next week: testing!